# 1. Training words: pseudo-writers rendered once

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/Meitei-Mayek-Diffusion-model/blob/main/notebooks/1_render_data.ipynb)

**Runtime: CPU** (*Runtime > Change runtime type > CPU*). No GPU is needed, and a GPU runtime
costs compute units for nothing here.

The word synthesiser of the word-recognition project composes TUMMHCD characters into words.
Here it writes as **pseudo-writers**: each writer keeps one word style (letter size and width,
spacing, slant, pen, how often letters join, whether ꯥ ꯩ ꯪ go beside their letter) and a small
set of images per character across its 16 words. Each word is kept on the 64-row canvas
(baseline at row 46, letters 22 px high) with its printed form stretched over its ink.

1. **Validation writers** (500), then a sheet of 8 of them (6 words each, the printed form
   below each word, paler). Look at it before the long part.
2. **Test writers** (1,000).
3. **Training writers** (75,000; 1.2 million words): about 4.3 core-hours, so about 2 hours on
   2 cores. Shards already written are kept: if the runtime disconnects, run the first two
   cells and this one again.
4. **Counts** to `results/data_stats.json`.

Inputs on Drive, from the word-recognition project (`WORK_OLD`): `glyphs/{train,val,test}.npz`
and `lexicon/{train,val,test}.tsv`. Outputs in `WORK/data/` (about 4.3 GB): keep at least
12 GB free on Drive for this project.

In [ ]:
import os

# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-diffusion"         # this project's working folder
WORK_OLD = "/content/drive/MyDrive/meitei-word-recognition"   # the word-recognition project's folder
REPO = "chingkheinganba231005/Meitei-Mayek-Diffusion-model"
BRANCH = "main"
WORD_REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
WORD_COMMIT = "bdee9e7"                                       # the synthesiser this project builds on
CONTENT = "/content"                                          # the runtime's own disk

WRITERS = {"train": 75000, "val": 500, "test": 1000}   # pseudo-writers per split
WORDS = 16                                             # words per writer
PROCESSES = os.cpu_count()                             # rendering processes

DISCONNECT_AT_END = True    # the last cell disconnects the runtime (Colab bills a connected runtime)

In [ ]:
import json, re, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
os.chdir(CONTENT)
CODE, WORD = f"{CONTENT}/code", f"{CONTENT}/word_repo"


def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")


def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails.
    Returns the output."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")
    return "".join(lines)


def clone(url, path, ref):
    """A fresh copy of a repository at a branch or a commit."""
    shutil.rmtree(path, ignore_errors=True)
    run("git", "clone", "-q", url, path)
    run("git", "-C", path, "checkout", "-q", ref)


clone(f"https://github.com/{REPO}.git", CODE, BRANCH)
clone(f"https://github.com/{WORD_REPO}.git", WORD, WORD_COMMIT)
sys.path[:0] = [CODE, WORD]
os.chdir(CODE)

for d in ("data", "runs", "results", "eval"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
run("git", "-C", CODE, "log", "-1", "--format=code: %h %s")

INPUTS = f"{CONTENT}/inputs"
needed = [f"glyphs/{s}.npz" for s in ("train", "val", "test")] + [f"lexicon/{s}.tsv" for s in ("train", "val", "test")]
missing = [f for f in needed if not os.path.exists(f"{WORK_OLD}/{f}")]
if missing:
    raise SystemExit("missing in WORK_OLD (from the word-recognition project):\n  " + "\n  ".join(missing))
for f in needed:
    os.makedirs(os.path.dirname(f"{INPUTS}/{f}"), exist_ok=True)
    if not os.path.exists(f"{INPUTS}/{f}"):
        shutil.copy(f"{WORK_OLD}/{f}", f"{INPUTS}/{f}")   # local copies read faster than Drive
print(f"{PROCESSES} processes; free on Drive: {shutil.disk_usage(WORK).free / 1e9:.0f} GB")


def render(split):
    stream(sys.executable, "-u", "scripts/render_data.py", "--word-repo", WORD, "--glyphs", f"{INPUTS}/glyphs",
           "--lexicon", f"{INPUTS}/lexicon", "--out", f"{WORK}/data", "--splits", split,
           "--writers", *[f"{k}={v}" for k, v in WRITERS.items()], "--words", WORDS, "--processes", PROCESSES)

## 1. Validation writers, and a look

Each row is one writer: six of its words, each above its printed form (paler), which the
generator receives as its content. A writer's words should share their size, slant, pen and
letter shapes. If something looks wrong, stop here.

In [ ]:
from IPython.display import Image as Show, display
from mayek_diffusion.data import WordSet
from mayek_diffusion.sheets import preview

render("val")
preview(WordSet(f"{WORK}/data/val"), f"{WORK}/eval/data_preview.png", writers=8, words=6)
display(Show(f"{WORK}/eval/data_preview.png"))

## 2. Test writers

In [ ]:
render("test")

## 3. Training writers

The long part. Each line is a shard of 2,000 writers. If the runtime disconnects, run the
first two cells and this one again: the shards already on Drive are kept.

In [ ]:
render("train")

## 4. Counts

In [ ]:
from mayek_diffusion.render import stats

counts = stats(f"{WORK}/data")
Path(f"{WORK}/results/data_stats.json").write_text(json.dumps(counts, indent=1))
for split, c in counts.items():
    print(f"{split}: {c['writers']:,} writers, {c['words']:,} words, canvas width p50/p99/max "
          f"{c['canvas_width']['50']:.0f}/{c['canvas_width']['99']:.0f}/{c['canvas_width']['max']:.0f} px, "
          f"{100 * c['writers_signs_beside']:.0f}% of writers put ꯥ ꯩ ꯪ beside, {c['megabytes']:,.0f} MB")

## Disconnect

Colab bills a connected runtime whether or not it is working. The cell below disconnects it
(`DISCONNECT_AT_END = True`); otherwise use *Runtime > Disconnect and delete runtime*. Every
result is already on Drive.

In [ ]:
# Colab bills a connected runtime whether or not it is working: disconnect it now.
if DISCONNECT_AT_END:
    from google.colab import runtime
    runtime.unassign()
else:
    print("Finished. Disconnect the runtime: Runtime > Disconnect and delete runtime.")